# 04 - Train/Test Split

Goal: build the modelable dataset by applying the EDA decisions, and split train/test before any encoding.

Here `Monthly_Charge < 0` is filtered out, structural missing values are imputed, leakage columns are excluded, and `Joined` is reserved for scoring.

In [1]:
import os

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = "../data/Customer_Data.csv"
PROCESSED_DIR = "../data/processed"

TARGET_COL = "Customer_Status"
CUSTOMER_ID_COL = "Customer_ID"
CHURN_CATEGORY_COL = "Churn_Category"
CHURN_REASON_COL = "Churn_Reason"
MONTHLY_CHARGE_COL = "Monthly_Charge"

INTERNET_DEPENDENT_COLS = [
    "Internet_Type",
    "Online_Security",
    "Online_Backup",
    "Device_Protection_Plan",
    "Premium_Support",
    "Streaming_TV",
    "Streaming_Movies",
    "Streaming_Music",
    "Unlimited_Data",
]

# Kept separate (not in LEAKAGE_COLS) because they aren't leakage in the strict sense --
# they are valid predictors for Stayed/Churned -- but they are out of range for Joined,
# which hasn't had time to accumulate these amounts yet (diagnosed with data in
# 03_statistical_analysis.ipynb).
CUMULATIVE_SINCE_SIGNUP_COLS = [
    "Total_Charges",
    "Total_Revenue",
    "Total_Refunds",
    "Total_Extra_Data_Charges",
    "Total_Long_Distance_Charges",
]

# Customer_ID doesn't contribute signal as a feature, but it's kept in df_joined to be
# able to identify each customer when scoring them in 09_business_insights.ipynb -- it's
# only dropped as a feature in 05_feature_engineering.ipynb. Churn_Category/Churn_Reason
# only exist for customers who already churned, they are a consequence of churn, not a
# predictor -- including them would be real leakage, and they don't apply to Joined
# (it doesn't have these columns populated).
LEAKAGE_COLS_MODEL = [CUSTOMER_ID_COL, CHURN_CATEGORY_COL, CHURN_REASON_COL]
LEAKAGE_COLS_JOINED = [CHURN_CATEGORY_COL, CHURN_REASON_COL]

df = pd.read_csv(DATA_PATH)
df_model = df[df[TARGET_COL].isin(["Stayed", "Churned"])].copy()
df_joined = df[df[TARGET_COL] == "Joined"].copy()

print(f"df_model  (Stayed/Churned, for training/evaluation): {df_model.shape[0]} rows")
print(f"df_joined (Joined, scored at the end in 09):         {df_joined.shape[0]} rows")

df_model  (Stayed/Churned, for training/evaluation): 6007 rows
df_joined (Joined, scored at the end in 09):         411 rows


## Filter out negative `Monthly_Charge` and impute structural missing values

In [2]:
negative_mask = df_model[MONTHLY_CHARGE_COL] < 0
negative_mask_joined = df_joined[MONTHLY_CHARGE_COL] < 0
df_model = df_model.loc[~negative_mask].copy()
df_joined = df_joined.loc[~negative_mask_joined].copy()
print(f"Rows excluded for negative Monthly_Charge -- df_model: {negative_mask.sum()}, df_joined: {negative_mask_joined.sum()}")

for frame in (df_model, df_joined):
    for col in INTERNET_DEPENDENT_COLS:
        frame[col] = frame[col].fillna("No Internet Service")
    frame["Multiple_Lines"] = frame["Multiple_Lines"].fillna("No Phone Service")
    frame["Value_Deal"] = frame["Value_Deal"].fillna("No Deal")

Rows excluded for negative Monthly_Charge -- df_model: 101, df_joined: 6


These rules don't learn parameters from the dataset: they are fixed cleaning decisions. That's why they can be applied before the split without introducing leakage.

Categorical encoding is left for `05_feature_engineering.ipynb`, fitted only on train.

## Exclude leakage columns and columns accumulated since sign-up

In [3]:
print(f"Leakage columns excluded from df_model: {LEAKAGE_COLS_MODEL}")
print(f"Leakage columns excluded from df_joined: {LEAKAGE_COLS_JOINED}")
print(f"Columns accumulated since sign-up excluded: {CUMULATIVE_SINCE_SIGNUP_COLS}")

df_model = df_model.drop(columns=LEAKAGE_COLS_MODEL + CUMULATIVE_SINCE_SIGNUP_COLS)
df_joined = df_joined.drop(columns=LEAKAGE_COLS_JOINED + CUMULATIVE_SINCE_SIGNUP_COLS)

Leakage columns excluded from df_model: ['Customer_ID', 'Churn_Category', 'Churn_Reason']
Leakage columns excluded from df_joined: ['Churn_Category', 'Churn_Reason']
Columns accumulated since sign-up excluded: ['Total_Charges', 'Total_Revenue', 'Total_Refunds', 'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges']


## Train/test split

`stratify=y` keeps the churn proportion consistent between train and test. `random_state=42` makes the split reproducible.

The split happens before encoding so that test doesn't influence the columns learned by the model.

In [4]:
X = df_model.drop(columns=[TARGET_COL])
y = (df_model[TARGET_COL] == "Churned").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} rows (churn rate {y_train.mean():.3f})")
print(f"Test:  {X_test.shape[0]} rows (churn rate {y_test.mean():.3f})")

Train: 4724 rows (churn rate 0.289)
Test:  1182 rows (churn rate 0.288)


## Save the unencoded datasets

In [5]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_raw = X_train.copy()
train_raw["churn_flag"] = y_train
train_raw.to_parquet(f"{PROCESSED_DIR}/train_raw.parquet", index=False)

test_raw = X_test.copy()
test_raw["churn_flag"] = y_test
test_raw.to_parquet(f"{PROCESSED_DIR}/test_raw.parquet", index=False)

joined_raw = df_joined.drop(columns=[TARGET_COL])
joined_raw.to_parquet(f"{PROCESSED_DIR}/joined_raw.parquet", index=False)

print("Saved: train_raw.parquet, test_raw.parquet, joined_raw.parquet")

Saved: train_raw.parquet, test_raw.parquet, joined_raw.parquet
